# 01 — Greeks Lab: Sensitivity Curves and Position Greeks

You will plot each greek as a function of spot and time, then aggregate greeks across a
two-leg position. Everything on DEMO: spot **$100**, IV **0.25**, mostly **45 DTE**.

Goals:
1. `greeks.bsm_greeks` — delta/gamma/theta/vega vs spot, and vs DTE.
2. `greeks.position_greeks` — dollar-aggregated greeks of real positions.
3. `viz.plot_greeks` — position greek curves.
4. See *delta-neutral, short-vega* in the numbers of a short straddle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import greeks, strategies, viz

SPOT, VOL = 100.0, 0.25
t = 45 / 365

## 1. The greeks of a single ATM call

`bsm_greeks` returns a `Greeks` dataclass (delta, gamma, theta, vega, rho), per share.

In [ ]:
g = greeks.bsm_greeks('call', SPOT, strike=100, t=t, vol=VOL)
print(f'delta {g.delta:.3f}  gamma {g.gamma:.4f}  theta {g.theta:.4f}  vega {g.vega:.3f}  rho {g.rho:.3f}')

## 2. Delta vs spot — from 0 to 1 for calls, 0 to -1 for puts

Sweep spot with the strike fixed at 100. Watch call delta rise through 0.5 ATM toward 1 ITM.

In [ ]:
spots = np.linspace(70, 130, 121)
call_delta = [greeks.bsm_greeks('call', s, 100, t, VOL).delta for s in spots]
put_delta  = [greeks.bsm_greeks('put',  s, 100, t, VOL).delta for s in spots]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(spots, call_delta, label='call delta')
ax.plot(spots, put_delta, label='put delta')
ax.axvline(100, color='k', ls='--', lw=1); ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('spot'); ax.set_ylabel('delta'); ax.set_title('Delta vs spot (strike 100, 45 DTE)'); ax.legend()
plt.show()

## 3. Gamma, theta, vega all peak at-the-money

Plot the three ATM-peaking greeks against spot. Note gamma and vega are positive (long option);
theta is negative (you pay decay).

In [ ]:
gam = [greeks.bsm_greeks('call', s, 100, t, VOL).gamma for s in spots]
the = [greeks.bsm_greeks('call', s, 100, t, VOL).theta for s in spots]
veg = [greeks.bsm_greeks('call', s, 100, t, VOL).vega  for s in spots]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, y, name in zip(axes, [gam, the, veg], ['gamma', 'theta', 'vega']):
    ax.plot(spots, y); ax.axvline(100, color='k', ls='--', lw=1); ax.set_title(name); ax.set_xlabel('spot')
plt.tight_layout(); plt.show()

## 4. Gamma explodes and vega fades as expiration approaches

Hold the ATM strike; shrink DTE. Gamma grows sharply near expiry; vega shrinks. This is the
seller's dilemma in one chart.

In [ ]:
dtes = np.array([90, 60, 45, 30, 21, 14, 7, 3, 1])
atm_gamma = [greeks.bsm_greeks('call', 100, 100, d/365, VOL).gamma for d in dtes]
atm_vega  = [greeks.bsm_greeks('call', 100, 100, d/365, VOL).vega  for d in dtes]
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(dtes, atm_gamma, 'o-', color='C1', label='gamma'); ax1.set_xlabel('DTE'); ax1.set_ylabel('gamma', color='C1')
ax2 = ax1.twinx(); ax2.plot(dtes, atm_vega, 's-', color='C0', label='vega'); ax2.set_ylabel('vega', color='C0')
ax1.invert_xaxis(); ax1.set_title('ATM gamma rises, vega falls into expiry'); plt.show()

## 5. Position greeks: a bull call spread

Build the DEMO 100/110 bull call spread and aggregate its greeks in **dollars**
(`quantity × multiplier` applied). Compare to the naked long 100 call.

In [ ]:
spread = strategies.bull_call_spread((100, 3.91), (110, 0.73), expiry=t)
lone   = strategies.long_call((100, 3.91), expiry=t)
gs = greeks.position_greeks(spread, SPOT, VOL)
gl = greeks.position_greeks(lone,   SPOT, VOL)
print('spread  ', f'delta {gs.delta:7.1f}  theta {gs.theta:7.2f}  vega {gs.vega:7.2f}')
print('long call', f'delta {gl.delta:7.1f}  theta {gl.theta:7.2f}  vega {gl.vega:7.2f}')

The spread has a **smaller delta** than the lone call (the short 110 leg subtracts direction) and
**less vega and less theta** — selling the 110 call trims the greeks you did not want to pay for.

## 6. Delta-neutral, short-vega, positive-theta: the short straddle

Sell the 100 call and 100 put. The deltas nearly cancel (delta-neutral); both legs are short, so
vega is negative (short vega) and theta positive (you collect decay).

In [ ]:
straddle = strategies.short_straddle((100, 3.91), (100, 3.42), expiry=t)
gstr = greeks.position_greeks(straddle, SPOT, VOL)
print(f'delta {gstr.delta:7.2f}  (near zero => delta-neutral)')
print(f'gamma {gstr.gamma:7.3f}  (negative => short gamma, danger on big moves)')
print(f'theta {gstr.theta:7.2f}  (positive => collect decay daily)')
print(f'vega  {gstr.vega:7.2f}  (negative => short vega, profit if IV falls)')

## 7. Position greek curves vs spot

`viz.plot_greeks` shows how the position's dollar greeks change as spot moves. For the short
straddle, watch delta cross zero at 100 and gamma stay negative everywhere.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_greeks(straddle, vol=VOL, which=('delta', 'gamma'), ax=ax)
ax.set_title('Short straddle: position delta & gamma vs spot')
plt.show()

## Experiments

1. In section 4, redo the DTE sweep for **theta** instead of vega. Confirm theta's magnitude
   grows into expiration — the mirror of the extrinsic decay from module 00.
2. In section 5, widen the spread to 100/120 (short the 120 call at ~0.07). What happens to the
   net delta and net vega versus the 100/110 version?
3. In section 6, move the straddle to a **strangle**: short the 95 put and 105 call. Is it still
   delta-neutral? How do gamma and theta compare to the straddle?
4. Re-run section 2 at 7 DTE instead of 45. How much *sharper* is the delta transition through
   the strike (the gamma effect)?
5. Compute `position_greeks` for a single **short** 100 put. Confirm its delta is positive and
   its vega negative — the mirror of a long put.